In [1]:
import numpy as np


class SpatialScalePriorAnchorGenerator:

    def __init__(self, img_shape=(640, 640), feature_stride=16):
        self.img_h, self.img_w = img_shape
        self.stride = feature_stride
        self.grid_h = self.img_h // feature_stride
        self.grid_w = self.img_w // feature_stride

    def fit_spatial_scale_kde(self, gt_boxes):
        """GT Bounding Box (cx, cy, w, h) 데이터로부터

        위치별(Grid Cell) 최적의 Scale/Aspect Ratio 분포를 Kernel Density로 추정.
        """
        # Normalize coordinates
        norm_boxes = gt_boxes.copy()
        norm_boxes[:, [0, 2]] /= self.img_w
        norm_boxes[:, [1, 3]] /= self.img_h

        # Grid Cell별로 속한 GT Box들의 (w, h) 통계 집계
        grid_priors = {}
        for gy in range(self.grid_h):
            for gx in range(self.grid_w):
                # Grid 중심 위치
                cell_cx = (gx + 0.5) / self.grid_w
                cell_cy = (gy + 0.5) / self.grid_h

                # Grid 중심과의 거리에 따른 가중치(Gaussian Kernel) 부여
                dists = np.sqrt(
                    (norm_boxes[:, 0] - cell_cx) ** 2
                    + (norm_boxes[:, 1] - cell_cy) ** 2
                )
                weights = np.exp(-(dists**2) / (2 * (0.15**2)))  # sigma = 0.15

                if np.sum(weights) > 1e-5:
                    # 가중 평균 기반 위치 맞춤형 (Width, Height) Prior 추정
                    avg_w = np.average(gt_boxes[:, 2], weights=weights)
                    avg_h = np.average(gt_boxes[:, 3], weights=weights)
                    grid_priors[(gy, gx)] = (avg_w, avg_h)
                else:
                    grid_priors[(gy, gx)] = (32.0, 32.0)  # Default fallback

        return grid_priors

    def generate_guided_anchors(self, grid_priors, probability_threshold=0.2):
        """공간 밀도가 높은 위치에만 맞춤형 앵커를 가변 생성 (Guided Anchoring 원리)"""
        guided_anchors = []
        for (gy, gx), (w, h) in grid_priors.items():
            cx = (gx + 0.5) * self.stride
            cy = (gy + 0.5) * self.stride

            # 위치별 특징(예: 도로 하단엔 차/사람 크기 앵커, 상단엔 작은 객체 앵커) 반영
            guided_anchors.append(
                {
                    "grid": (gy, gx),
                    "center": (cx, cy),
                    "dynamic_anchor_size": (round(w, 1), round(h, 1)),
                }
            )
        return guided_anchors


# === 실험 실행 ===
# 1. 자율주행 시나리오 가상 데이터: 중앙/하단에 차·보행자 집중 분포 (Center & Horizon Bias)
np.random.seed(42)
num_samples = 1000
cx_samples = np.random.normal(loc=320, scale=120, size=num_samples)
cy_samples = np.random.normal(
    loc=400, scale=80, size=num_samples
)  # 하단 지평선 근처
w_samples = (cy_samples / 640.0) * np.random.normal(
    loc=150, scale=20, size=num_samples
)  # 가까울수록(하단) 더 큼
h_samples = w_samples * np.random.uniform(0.8, 1.8, size=num_samples)

gt_boxes = np.column_stack([cx_samples, cy_samples, w_samples, h_samples])
gt_boxes = np.clip(gt_boxes, 10, 630)

# 2. 공간-크기 결합 추정 앵커 생성기 작동
generator = SpatialScalePriorAnchorGenerator(
    img_shape=(640, 640), feature_stride=64
)
priors = generator.fit_spatial_scale_kde(gt_boxes)
anchors = generator.generate_guided_anchors(priors)

print("--- [2D 공간 위치별 동적 제안 앵커 샘플] ---")
for a in anchors[::15]:  # 일부 그리드 출력
    print(
        f"Grid Position {a['grid']} | Center: {a['center']} -> Adaptive Anchor (W, H): {a['dynamic_anchor_size']}"
    )

--- [2D 공간 위치별 동적 제안 앵커 샘플] ---
Grid Position (0, 0) | Center: (32.0, 32.0) -> Adaptive Anchor (W, H): (np.float64(57.1), np.float64(74.4))
Grid Position (1, 5) | Center: (352.0, 96.0) -> Adaptive Anchor (W, H): (np.float64(65.3), np.float64(83.5))
Grid Position (3, 0) | Center: (32.0, 224.0) -> Adaptive Anchor (W, H): (np.float64(78.3), np.float64(100.4))
Grid Position (4, 5) | Center: (352.0, 288.0) -> Adaptive Anchor (W, H): (np.float64(83.9), np.float64(107.9))
Grid Position (6, 0) | Center: (32.0, 416.0) -> Adaptive Anchor (W, H): (np.float64(97.0), np.float64(126.1))
Grid Position (7, 5) | Center: (352.0, 480.0) -> Adaptive Anchor (W, H): (np.float64(101.9), np.float64(131.5))
Grid Position (9, 0) | Center: (32.0, 608.0) -> Adaptive Anchor (W, H): (np.float64(113.7), np.float64(145.1))
